# Sessions: what the archive of nights says about drift and the worm

Across every archived log (`logs/archive`, registered in `utility/sessions.toml`). One night at a time is
`analyse_tracking.ipynb`; the worm profile test is `analyse_worm_profile.ipynb`. The questions here:

1. **What is in the archive?** Every steady-tracking segment, its pose and how fast each motor turned.
2. **Which PEC drift time constant suits most nights?** The driver's EMA replayed on every segment's drift.
3. **Is there a worm, and is it on the motor angle?** Periodic error on each motor's angle vs on time, and which
   worm angle explains it (the evidence for the 6.0° worm the feed-forward corrects).
4. **Does each motor's worm phase repeat night to night?** Needed for a fixed profile on the MCU angles.
5. **Is the worm steady through a night?**
6. **Findings.**

**Data.** Each catalogued segment (`catalog_logs.py` → `extract_segments.py`) gives the mount's drift (the guide
corrections + PEC applied, so PEC's own effect is removed, joined across PEC resets; arcsec, the driver's correction
convention) and the motor angles on the same time base. Segments are rebuilt when `sessions.toml` changes.

**Scores.** PEC applies a rate continuously and the guider removes what is left at each correction, so a PEC setting
is scored by the *drift left for the guider* over a guide interval dt: `y(t + dt) − y(t) − rate(t) × dt`, pooled RMS,
relative to no PEC (1.0; lower is better). The driver's own `PecAxis` is replayed causally: what it would have learnt
at each moment.

In [ ]:
import os, sys, glob, re
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
sys.path.insert(0, os.path.abspath(os.path.join('.', '..', 'driver')))
import importlib, catalog_logs, extract_segments, pe_analysis, sessions
for m in (sessions, pe_analysis, catalog_logs, extract_segments):     # dependencies first: extract_segments imports pe_analysis
    importlib.reload(m)
from catalog_logs import catalog
from extract_segments import extract_all, DEFAULT_SITE
from sessions import catalog_notes, catalog_phd2
from pe_analysis import (load_segments, benchmark_segments, pooled_relative, PecModel, theta_scan, time_scan, MOTORS,
                         WORM_THETA, worm_benchmark, relative_to, worm_angle_control, worm_phase_table,
                         phase_consistency, windowed_worm)

LOG_DIR = os.path.abspath(os.path.join('..', 'logs', 'archive'))
SEG_DIR = os.path.join(LOG_DIR, 'segments')
CATALOG_CSV = os.path.join(LOG_DIR, 'catalog_segments.csv')
_INDEX = os.path.join(SEG_DIR, 'index.csv')
REBUILD = (not os.path.exists(_INDEX)                                       # or set True to force
           or os.path.getmtime(sessions.DEFAULT_REGISTRY) > os.path.getmtime(_INDEX))
MIN_MINUTES = 60          # segments used for the drift and worm questions
TEMPLATE = 'plotly_dark'
COLOURS = {'M1': 'royalblue', 'M2': 'mediumseagreen', 'M3': 'orange'}

OWNER = lambda session: next((o for o in ('mark', 'greg', 'vyskocil') if o in session), 'jdm')   # whose mount
_lat = {}
def lat_of(session):
    # the site latitude a client set during the session (its last SiteLatitude), else the default site
    if session not in _lat:
        found = []
        for p in glob.glob(os.path.join(LOG_DIR, f'{session}*.log')):
            with open(p, errors='replace') as f:
                found += re.findall(r"'SiteLatitude': '(-?[\d.]+)'", f.read())
        _lat[session] = float(found[-1]) if found else DEFAULT_SITE[0]
    return _lat[session]

## 1. What is in the archive?

One row per steady-tracking segment (split at gotos, slews, jogs, rotates, parks and tracking changes; first 60 s
skipped). `usable` = at least 45 min with a pose source. Then each motor's mean rate per segment: a worm on a motor
repeats every 6° / rate, so segments where one motor dominates, and the same motor at different rates, are what tell a
worm from a time-periodic error.

In [ ]:
if REBUILD or not os.path.exists(CATALOG_CSV):
    cat = catalog(LOG_DIR, catalog_notes())     # notes from sessions.toml
    cat.to_csv(CATALOG_CSV, index=False)
cat = pd.read_csv(CATALOG_CSV, parse_dates=['start', 'end'])
use = cat[cat['usable']].copy()
print(f"{len(cat)} segments in {cat['session'].nunique()} sessions; {len(use)} usable, {use['duration_min'].sum() / 60:.1f} h")
cols = ['session', 'segment', 'start', 'duration_min', 'driver_version', 'pec', 'n_sync_guide', 'sync_interval_s',
        'pulse_guiding', 'pose_source', 'alt_mean', 'roll_mean', 'm1_dps_hr', 'm2_dps_hr', 'm3_dps_hr', 'notes']
show = use[cols].copy()
show['session'] = show['session'].str.replace('alpaca.', '', regex=False)
show['start'] = show['start'].dt.strftime('%Y-%m-%d %H:%M')
display(show.round(1))

use['n_fast'] = (use[['m1_dps_hr', 'm2_dps_hr', 'm3_dps_hr']] >= 8).sum(axis=1)
fig = make_subplots(rows=1, cols=3, subplot_titles=['M1 vs M3 rate', 'M2 vs M3 rate', 'Hours by number of fast motors (>= 8 deg/hr)'])
hover = use['session'].str.replace('alpaca.', '', regex=False) + ' #' + use['segment'].astype(str)
fig.add_trace(go.Scatter(x=use['m1_dps_hr'], y=use['m3_dps_hr'], mode='markers', text=hover,
                         marker=dict(size=np.sqrt(use['duration_min']) * 1.5, color=use['alt_mean'], colorscale='Viridis',
                                     colorbar=dict(title='alt', x=0.3)), name='segments'), row=1, col=1)
fig.add_trace(go.Scatter(x=use['m2_dps_hr'], y=use['m3_dps_hr'], mode='markers', text=hover, showlegend=False,
                         marker=dict(size=np.sqrt(use['duration_min']) * 1.5, color=use['alt_mean'], colorscale='Viridis',
                                     showscale=False)), row=1, col=2)
h = use.groupby('n_fast')['duration_min'].sum() / 60
fig.add_trace(go.Bar(x=h.index.astype(str), y=h.values, showlegend=False), row=1, col=3)
fig.update_xaxes(title_text='M1 deg/hr', row=1, col=1); fig.update_yaxes(title_text='M3 deg/hr', row=1, col=1)
fig.update_xaxes(title_text='M2 deg/hr', row=1, col=2); fig.update_xaxes(title_text='fast motors', row=1, col=3)
fig.update_layout(height=450, width=1400, template=TEMPLATE, title='Segment motor rates (marker size ~ duration, colour = altitude)')
fig.show()

if REBUILD:
    extract_all(CATALOG_CSV, LOG_DIR, SEG_DIR, catalog_phd2())
idx = pd.read_csv(_INDEX)
SRC = dict(zip(idx['session'] + '#' + idx['segment'].astype(int).astype(str), idx['theta_source']))
RAW = {k for k, v in SRC.items() if v in ('kflog', 'peclog_raw', 'peclog_zeta', 'sglog')}   # raw MCU motor angles
segs = load_segments(SEG_DIR, min_minutes=MIN_MINUTES)
raw_segs = [s for s in segs if s.name in RAW]
print(f"{len(idx)} segment datasets; {len(segs)} of >= {MIN_MINUTES} min from {len({s.session for s in segs})} sessions "
      f"({sum(s.hours for s in segs):.0f} h), {len(raw_segs)} with raw motor angles")
print(idx.groupby(['drift_source', 'theta_source']).size().rename('segments').to_string())

## 2. Which PEC drift time constant suits most nights?

The EMA (`PecAxis`) replayed on every segment with a range of taus, scored separately for sync-guided segments (drift
sampled at plate solves) and pulse-guided ones (sampled every few seconds). A short tau follows changes but passes
noise; a long one lags. Then each segment's own best tau, and what one global tau loses against it. The driver
default is 7.5 min (`pec_tau_sec = 450`).

In [ ]:
GUIDE_INTERVALS = (30, 120, 300)                # s: PHD2-ish, typical plate solve, slow plate solve
TAUS_MIN = (1, 2, 5, 7.5, 10, 15, 21, 30, 45, 60)
MODELS = [PecModel(f'EMA tau {t:g}m', t * 60) for t in TAUS_MIN]
bench = benchmark_segments(segs, MODELS, guide_intervals_s=GUIDE_INTERVALS, ref_half_windows_s=(150, 300))
order = ['no_pec'] + [m.name for m in MODELS]
cols = [f'guide residual {dt}s' for dt in GUIDE_INTERVALS]
tables = {}
for kind in ('sync', 'pulse', 'all'):
    g = bench if kind == 'all' else bench[bench['kind'] == kind]
    if g.empty:
        continue
    tables[kind] = pooled_relative(g).reindex(order)[cols]
    print(f"{kind}: {g['seg'].nunique()} segments")
display(pd.concat(tables, axis=1).round(3))

fig = go.Figure()
for dt, c in zip(GUIDE_INTERVALS, ('royalblue', 'orange', 'mediumseagreen')):
    col = f'guide residual {dt}s'
    fig.add_trace(go.Scatter(x=[m.tau_s / 60 for m in MODELS], y=[tables['all'].loc[m.name, col] for m in MODELS],
                             mode='lines+markers', name=f'{dt} s interval', line=dict(color=c)))
fig.add_hline(y=1.0, line=dict(color='gray'))
fig.add_vline(x=7.5, line=dict(color='gray', dash='dot'))
fig.update_xaxes(type='log', title='EMA tau (minutes; dotted: driver default)')
fig.update_yaxes(title='drift left for the guider, relative to no PEC')
fig.update_layout(height=450, width=1000, template=TEMPLATE, title='EMA tau sweep, all segments')
fig.show()

INTERVAL = 120
g = bench[(bench['metric'] == f'guide residual {INTERVAL}s') & (bench['model'] != 'no_pec')].copy()
g['tau_min'] = g['model'].map({m.name: m.tau_s / 60 for m in MODELS})
per_seg = g.groupby(['seg', 'tau_min'])['value'].apply(lambda v: np.sqrt((v ** 2).mean())).reset_index()
best = per_seg.loc[per_seg.groupby('seg')['value'].idxmin()].set_index('seg')
print(f"best tau per segment ({INTERVAL} s interval):", best['tau_min'].value_counts().sort_index().to_dict())
loss = pd.DataFrame({f'{tau:g} min': (per_seg[per_seg['tau_min'] == tau].set_index('seg')['value'] / best['value'] - 1) * 100
                     for tau in TAUS_MIN})
display(loss.describe().loc[['50%', 'mean', 'max']].rename(index={'50%': 'median loss %', 'mean': 'mean loss %',
                                                                  'max': 'worst loss %'}).round(1))

## 3. Is there a worm, and is it on the motor angle?

* **Angle scan vs time scan:** for each motor and candidate worm angle, the mean periodogram power of the drift at
  the period that worm gives in each segment (angle / that motor's rate), against the power at one fixed time period
  in every segment. A purely time-based error can also make an angle peak (each segment hits *some* angle =
  period × rate), so the two are compared with each other, not read alone.
* **Worm angle controls:** EMA plus a worm profile fitted on each motor's angle, for each candidate worm angle (the
  tooth counts 960:1 allows, the old 54-tooth guess, arbitrary angles). *Hindsight*: fitted on the scored segment
  itself (extra parameters always help a little). *Cross-night*: learnt on the same mount's other nights and scored on
  a held-out night (raw motor angles only): extra parameters can't help there, so a dip at one angle and not elsewhere
  is a real worm. Slow: a few minutes.

In [ ]:
THETAS = np.arange(3.0, 12.01, 0.25)
PERIODS = np.arange(10, 80.1, 1) * 60
angle = theta_scan(segs, THETAS)
times = time_scan(segs, PERIODS)
fig = make_subplots(rows=1, cols=2, subplot_titles=['Angle scan: power at worm angle / motor rate', 'Time scan: power at a fixed period'])
for m in MOTORS:
    fig.add_trace(go.Scatter(x=angle.index, y=angle[m], name=m, line=dict(color=COLOURS[m])), row=1, col=1)
fig.add_vline(x=WORM_THETA, line=dict(color='gray', dash='dot'), row=1, col=1)
fig.add_trace(go.Scatter(x=times.index / 60, y=times['power'], name='time', line=dict(color='white')), row=1, col=2)
fig.update_xaxes(title_text='worm angle, deg of motor rotation (dotted: 6.0)', row=1, col=1)
fig.update_xaxes(title_text='period, min', row=1, col=2)
fig.update_yaxes(title_text='mean power', row=1, col=1)
fig.update_layout(height=430, width=1400, template=TEMPLATE)
fig.show()
summary = pd.DataFrame({m: {'peak': angle[m].idxmax(), 'peak power': angle[m].max(), 'median': angle[m].median()} for m in MOTORS}).T
summary.loc['time (min)'] = {'peak': times['power'].idxmax() / 60, 'peak power': times['power'].max(), 'median': times['power'].median()}
summary['peak / median'] = summary['peak power'] / summary['median']
display(summary.round(3))

In [ ]:
EMA = PecModel('EMA tau 7.5m', 450)                       # the driver default
SAME_MOUNT = lambda a, b: OWNER(a) == OWNER(b)
CANDIDATES = {f'{n} teeth': 360 / n for n in (30, 32, 40, 48, 60, 64, 80, 96)}
CANDIDATES.update({'54 teeth (972 guess)': 360 / 54, 'arbitrary 5.2': 5.2, 'arbitrary 6.9': 6.9,
                   'arbitrary 8.3': 8.3, 'arbitrary 10.3': 10.3})
ctl_hind = worm_angle_control(segs, tuple(CANDIDATES.values()), EMA, lat_of)
rows = []
for label, W in CANDIDATES.items():
    b = worm_benchmark(raw_segs, EMA, lat_of, raw_theta=RAW, same_mount=SAME_MOUNT, guide_intervals_s=(120,), worm_theta=W)
    b = b[b['model'].isin(['no_pec', EMA.name, 'worm: other sessions'])]
    per = b.pivot_table(index=['seg', 'axis'], columns='model', values='rms')
    v = (per['worm: other sessions'] / per[EMA.name]).dropna()
    rows.append({'worm': label, 'angle deg': round(W, 3), 'hindsight / EMA': ctl_hind[W],
                 'cross-night / EMA (pooled)': relative_to(b, EMA.name).loc['worm: other sessions', (120, 'vs ref')],
                 'cross-night median': v.median(), 'cross-night better in %': (v < 1).mean() * 100})
controls = pd.DataFrame(rows).sort_values('angle deg')
fig = go.Figure()
for col, c in (('hindsight / EMA', 'orange'), ('cross-night / EMA (pooled)', 'mediumseagreen')):
    fig.add_trace(go.Scatter(x=controls['angle deg'], y=controls[col], mode='markers+lines', name=col,
                             text=controls['worm'], line=dict(color=c)))
fig.add_hline(y=1.0, line=dict(color='gray', dash='dot'))
fig.update_layout(height=420, width=1100, template=TEMPLATE, xaxis_title='assumed worm angle (deg)',
                  yaxis_title='drift left for the guider, relative to EMA (120 s)', title='Worm angle controls')
fig.show()
display(controls.set_index('worm').round(3))

## 4. Does each motor's worm phase repeat night to night?

A fixed profile on the MCU angles works only if each motor's worm phase is the same every session, i.e. the MCU's
angle reference survives power cycles. For every raw-angle segment of this mount where a motor turns at least 2 worm
turns: the 1st-harmonic amplitude and phase from a joint fit of that segment. Per motor, how tightly the phases
cluster across sessions: R = 1 identical, ~0 random; p = the chance of that much clustering from random phases.
Control: the same at arbitrary worm angles, where phases should not cluster.

Most archived nights logged the 518 angles (box attitude × motor angles): M1's includes that night's heading, so its
phase can shift between sites or setups; nights with `zeta` (517) logged don't have that problem. The worm profile
test measures on 517 directly.

In [ ]:
PHASE_ANGLES = (WORM_THETA, 5.2, 6.9, 8.3, 4.5)                  # 6.0 and arbitrary controls
MOUNT = 'jdm'
for_mount = [s for s in raw_segs if OWNER(s.session) == MOUNT]
tables = {W: worm_phase_table(for_mount, lat_of, worm_theta=W) for W in PHASE_ANGLES}
ph = tables[WORM_THETA]
summary = pd.DataFrame({(f'{W:g} deg', m): phase_consistency(g.loc[g['amp'] >= 10, 'phase'])
                        for W, t in tables.items() for m, g in t.groupby('motor')}).T
print(f"{MOUNT}'s mount: phase clustering per motor (segments with amplitude >= 10\"):")
display(summary.round(3))
fig = make_subplots(rows=1, cols=3, specs=[[{'type': 'polar'}] * 3], subplot_titles=list(MOTORS))
for k, m in enumerate(MOTORS):
    g = ph[ph['motor'] == m]
    fig.add_trace(go.Scatterpolar(theta=(90 - g['phase']) % 360, r=g['amp'], mode='markers+text', marker=dict(size=10, color=COLOURS[m]),
                                  text=g['session'].str.replace('alpaca.', '').str[:22], textfont=dict(size=8),
                                  name=m, showlegend=False), row=1, col=k + 1)
fig.update_layout(height=480, width=1400, template=TEMPLATE,
                  title=f'{WORM_THETA:g} deg worm: 1st-harmonic peak angle (worm phase of the peak) and amplitude (arcsec, radius) per segment')
fig.show()
display(ph.assign(seg=ph['seg'].str.replace('alpaca.', '')).round(1))

## 5. Is the worm steady through a night?

A fixed profile assumes each motor's worm keeps its size and phase all night. One segment (`SEGMENT`, default the
longest with raw motor angles) cut into overlapping windows; in each, the three motors' worms fitted together. Flat
lines = a steady worm a fixed profile can correct; a trend or jump = it changes (pose, load). Hollow markers: M1 and
M3 turn at about the same speed in that window, so the fit can trade one for the other. A motor is fitted in a window
only if it turns at least 1.5 worm turns (9°) in it, so slow motors (often M2) can be missing.

In [ ]:
WINDOW_MIN, STEP_MIN = 90, 15
SEGMENT = None                                             # a segment name, or None for the longest raw-angle one
s = next(x for x in segs if x.name == SEGMENT) if SEGMENT else max(raw_segs, key=lambda x: x.hours)
w = windowed_worm(s, lat_of(s.session), window_s=WINDOW_MIN * 60, step_s=STEP_MIN * 60, worm_theta=WORM_THETA)
print(f"{s.name.replace('alpaca.', '')}: {s.hours:.1f} h, {SRC.get(s.name, '?')} angles, windows of {WINDOW_MIN} min every {STEP_MIN} min")
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['worm amplitude (arcsec of motor angle, 1st harmonic)', 'peak angle (worm phase of the peak, deg)'])
for m in MOTORS:
    g = w[(w['motor'] == m) & w['fitted']]
    if g.empty:
        continue
    hollow = ~g['m1_m3_separable'] & g['motor'].isin(['M1', 'M3'])
    g = g.assign(peak=(90 - g['phase']) % 360)
    for row, col in ((1, 'amp'), (2, 'peak')):
        fig.add_trace(go.Scatter(x=g['t_min'], y=g[col], mode='lines+markers', line=dict(color=COLOURS[m]),
                                 marker=dict(symbol=np.where(hollow, 'circle-open', 'circle'), size=8),
                                 name=m, legendgroup=m, showlegend=row == 1,
                                 text=[f'{x:.1f} turns, {r:.0f} deg/hr' for x, r in zip(g['turns'], g['rate'])]), row=row, col=1)
fig.update_yaxes(range=[0, 360], row=2, col=1)
fig.update_xaxes(title_text='minutes from segment start', row=2, col=1)
fig.update_layout(height=650, width=1300, template=TEMPLATE, title='Worm through the night (sliding windows)')
fig.show()

## 6. Findings (2026-10-05)

**PEC drift (section 2).**
- A rate-only EMA cuts the drift left for the guider by ~20–40 % depending on the interval (more for longer
  intervals, i.e. sync guiding). The best tau is about 5–10 min with a broad minimum; per segment it is mostly 2–5
  min, and a global 5 min loses ~3 % median against each segment's own best. **The default is EMA tau 7.5 min**:
  inside the minimum, and less noisy when the only input is a plate solve every 1–3 min.
- Removed after this comparison: RLS with time harmonics (as good as no PEC on sync-guided nights; the harmonics
  predict wrong as often as right), and a lag-compensated double EMA (helps only at long tau, to about where plain
  EMA already is). No time cycle is consistent enough to anticipate.

**The worm (sections 3–5).**
- Each motor has a 6.0° worm (60 teeth; 960:1 = 16 × 60). Only 6.0° (and 12°, whose 2nd harmonic is 6.0°)
  beats EMA cross-night; every other tooth count, the 54-tooth 6.67° and arbitrary angles score 1.00–1.07 (cross-night pooled).
- Its phase repeats night to night on the MCU angles (M2 and M3 R ≈ 0.95–0.99 on 518 angles; M1 R ≈ 0.86, but its
  518 angle carries the heading). The driver therefore corrects it with a fixed profile on 517 angles (the worm
  feed-forward), measured by the worm profile test rather than learnt from tracking: tracking turns M2 and M3 too
  slowly (often under 2 worm turns a night) to measure them well.
- First profile night (2026-10-04, `analyse_tracking.ipynb` section 7): the logged feed-forward matches the profile
  exactly; the guide corrections show M3 was under-corrected (profile 99″, needed ~125–150″), M1's phase off
  (225° applied vs ~245–285°), and M2 has a real 2nd harmonic of ~20″. The combined test (all motors, 2-D fit, both
  harmonics, pooled) replaces the per-motor tests that measured it.

**Open:**
- Does the worm repeat across sites / SPA alignments on 517 angles (cross-night profile tests)?
- What is the non-periodic drift PEC follows (alignment model residuals, backlash at reversals)?